In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install datasets -q

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping
import pickle
from datasets import load_dataset

# ==========================================
# 1. LOAD BUILT-IN DATASET (CNN/DailyMail)
# ==========================================
print("Downloading CNN/DailyMail dataset from Hugging Face...")
# We only grab the first 15,000 rows of the training set to save memory and time
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:15000]")

# Convert it to a Pandas DataFrame so it works perfectly with our code
df = dataset.to_pandas()

# The columns in this dataset are called 'article' and 'highlights'
text_col = 'article'       
summary_col = 'highlights' 

# Clean up any empty rows just in case
df.dropna(subset=[text_col, summary_col], inplace=True)

# Add <start> and <end> tokens to the target summaries
df[summary_col] = df[summary_col].apply(lambda x: '<start> ' + str(x) + ' <end>')
print("Dataset loaded successfully!")

# ==========================================
# 2. TOKENIZATION & PADDING (MEMORY SAFE)
# ==========================================
MAX_VOCAB_SIZE = 8000
MAX_TEXT_LEN = 80     # Articles get cut off after 80 words
MAX_SUMMARY_LEN = 15  # Summaries get cut off after 15 words

print("Tokenizing data...")
# Article Text (Encoder Input)
text_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
text_tokenizer.fit_on_texts(df[text_col])
text_vocab_size = min(len(text_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Summary Text (Decoder Input/Target)
summary_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
summary_tokenizer.fit_on_texts(df[summary_col])
summary_vocab_size = min(len(summary_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Convert to sequences
encoder_input_data = pad_sequences(text_tokenizer.texts_to_sequences(df[text_col]), maxlen=MAX_TEXT_LEN, padding='post', truncating='post')
decoder_input_data = pad_sequences(summary_tokenizer.texts_to_sequences(df[summary_col]), maxlen=MAX_SUMMARY_LEN, padding='post', truncating='post')

# Target data (shifted by 1 timestep)
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, 0:-1] = decoder_input_data[:, 1:]

# Explicitly cast to int32 to prevent the "Eager Execution" memory crash
encoder_input_data = np.array(encoder_input_data, dtype=np.int32)
decoder_input_data = np.array(decoder_input_data, dtype=np.int32)
decoder_target_data = np.array(decoder_target_data, dtype=np.int32)

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
latent_dim = 128 
embedding_dim = 100

# Encoder
encoder_inputs = Input(shape=(MAX_TEXT_LEN,), name='encoder_inputs')
enc_emb = Embedding(text_vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_SUMMARY_LEN,), name='decoder_inputs')
dec_emb_layer = Embedding(summary_vocab_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(summary_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Compile
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. TRAINING 
# ==========================================
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Starting custom summarizer training...")
model.fit(
    x=[encoder_input_data, decoder_input_data], 
    y=decoder_target_data,
    batch_size=64,
    epochs=50, 
    validation_split=0.2,
    callbacks=[early_stop]
)

# ==========================================
# 5. SAVE FILES FOR FRONTEND
# ==========================================
model.save('custom_summarizer_model.keras')

frontend_data = {
    'text_tokenizer': text_tokenizer,
    'summary_tokenizer': summary_tokenizer,
    'max_text_len': MAX_TEXT_LEN,
    'max_summary_len': MAX_SUMMARY_LEN,
    'latent_dim': latent_dim
}

with open('summarizer_tokenizer_data.pkl', 'wb') as f:
    pickle.dump(frontend_data, f)
    
print("\nSuccess! Download 'custom_summarizer_model.keras' and 'summarizer_tokenizer_data.pkl'.")

2026-03-22 14:18:52.128473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774189132.382171      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774189132.451338      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774189133.008976      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774189133.009025      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774189133.009028      55 computation_placer.cc:177] computation placer alr

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

Dataset loaded successfully!
Tokenizing data...


I0000 00:00:1774189187.726552      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774189187.732541      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Starting custom summarizer training...
Epoch 1/50


I0000 00:00:1774189192.138768     664 cuda_dnn.cc:529] Loaded cuDNN version 91002


188/188 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step - accuracy: 0.1663 - loss: 7.3107 - val_accuracy: 0.2121 - val_loss: 6.3139
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2207 - loss: 6.2474 - val_accuracy: 0.1908 - val_loss: 6.2872
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2216 - loss: 6.1785 - val_accuracy: 0.2017 - val_loss: 6.2021
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2297 - loss: 6.1162 - val_accuracy: 0.2017 - val_loss: 6.1635
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2289 - loss: 6.0810 - val_accuracy: 0.2017 - val_loss: 6.2038
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2287 - loss: 6.0683 - val_accuracy: 0.2017 - val_loss: 6.2029
Epoch 7/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2301 - loss: 6.0382 - val_accuracy: 0.2019 - val_loss: 6.0906
Epoch 8/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.2294 - loss: 6.0210 - val_accuracy: 0.21